# 07 · Out-of-fold cross-encoder scores (input of Direction 3)

**Goal:** cross-encoder scores for every **train** pair from a model that never saw that pair's query, so the LightGBM re-ranker (Direction 3) is trained on scores that behave like the ones it will see on val and test.

**Configuration** = the one chosen in Directions 1 and 2: DeBERTa-v3-base, `main`, `max_len` 256, **ce4**, lr 2e-5, 1,200 warm-up steps, 1 epoch, batch 32, fp16, seed 42. ce4 ties with MSE on val (D2: +0.0014 [−0.0013; 0.0042]) and also gives **p_E, p_S, p_C, p_I**, saved next to the score as re-ranker features.

**Folds:** 5 folds, `fold = fold10 // 2` (by `dup_group`, balanced pairs; README Conventions). Model *k* trains on the 4 other folds (~302k pairs, ~9,450 steps) and predicts:
- `oof`: its held-out fold *k* (scored: train labels, not a tuning set);
- `val`: scored, and compared with `d2_deberta_main_ce4_s42` (same configuration on all train) to see what 80% of train costs;
- `test`: saved blind.

For the re-ranker, val and test use the **mean of the 5 fold models** (section 3), so train, val and test scores all come from models trained on 80% of train.

**Kaggle settings:** GPU **T4**, **Internet on**. About **2.4 h per fold** (1.9 h training + 0.5 h prediction), so 2 folds per version (~5 h) stay well inside the 12 h session limit.

**Inputs:**
- Dataset `wm-src`, version whose `predict` has `return_probs` and whose `data/results/` has `d2_deberta_main_ce4_s42`. Required.
- Add Input → Notebook Output → **01_preprocess** (`pairs.parquet`, `products_clean.parquet`). Required.

**How to run.**
1. `MODE = 'smoke'` (fold 0, 5% of its training queries, 10% of the held-out queries, val; ~20 min). Check the verdict at the end of section 2.
2. `MODE = 'full'`, set `FOLDS` (e.g. `[0, 1]`), **Save Version → Save & Run All**. One version per group of folds; the two Kaggle accounts can run different folds at the same time. After each version, download `results/`.
3. `MODE = 'assemble'` (CPU, no GPU needed) with the outputs of all 5 folds attached as inputs: writes `oof/ce_deberta_main_ce4.parquet`.

**Output** (`/kaggle/working/results/`): `oof_deberta_main_ce4_f<k>.json` and `scores/oof_deberta_main_ce4_f<k>_{oof,val,test}.parquet` (`example_id, query_id, score, p_E, p_S, p_C, p_I`); after `assemble`: `oof/ce_deberta_main_ce4.parquet`.

In [ ]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

In [ ]:
import gc, inspect, os, time

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; before torch

import numpy as np
import pandas as pd

from src.data import load_pairs
from src.evaluation import RunLog, summarize

MODE = 'smoke'    # 'smoke': fold 0 on 5% of train, val only | 'full': the folds in FOLDS | 'assemble': CPU, join 5 folds
FOLDS = [0, 1]    # full mode: folds trained in this version (0-4); ~2.4 h each on a T4

CFG = {'model': 'microsoft/deberta-v3-base', 'fields': 'main', 'max_len': 256, 'loss': 'ce4',
       'lr': 2e-5, 'warmup_steps': 1200, 'epochs': 1, 'batch_size': 32, 'weight_decay': 0.01,
       'max_grad_norm': 1.0, 'fp16': True, 'eval_batch_size': 64}           # = d2_deberta_main_ce4_s42
SEED = 42
N_FOLDS = 5                                                                  # fold = fold10 // 2
NAME = 'oof_deberta_main_ce4'
REF = 'd2_deberta_main_ce4_s42'                                              # same config trained on all train
PROBS = ['p_E', 'p_S', 'p_C', 'p_I']
RES = Path('/kaggle/working/results')

assert MODE in ('smoke', 'full', 'assemble')
if MODE == 'smoke':
    FOLDS = [0]
assert set(FOLDS) <= set(range(N_FOLDS))
if MODE != 'assemble':
    import torch
    import transformers
    from src.cross_encoder import predict, train_ce
    assert 'return_probs' in inspect.signature(predict).parameters, \
        'wm-src is an old version: rebuild the zip (predict with return_probs) and update the input'
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    transformers.logging.set_verbosity_error()
    assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4'
    print('torch', torch.__version__, '| transformers', transformers.__version__, '|', torch.cuda.get_device_name(0))
print('MODE', MODE, '| FOLDS', FOLDS if MODE != 'assemble' else 'all')

## 1. Data and folds

All pairs with the `main` text. Train pairs get `fold = fold10 // 2`; every fold should hold about 20% of the train pairs and queries.

In [ ]:
pairs = load_pairs(CFG['fields'])
pairs['fold'] = np.where(pairs['split'] == 'train', pairs['fold10'] // 2, -1)
train_all = pairs[pairs['split'] == 'train']
fixed = {s: pairs[pairs['split'] == s] for s in ['val', 'test']}

sizes = train_all.groupby('fold').agg(pairs=('example_id', 'size'), queries=('query_id', 'nunique'),
                                      share_E=('esci_label', lambda l: round((l == 'E').mean(), 4)))
display(sizes)
assert len(sizes) == N_FOLDS and train_all.groupby('query_id')['fold'].nunique().max() == 1   # a query sits in one fold
print(f"val {len(fixed['val']):,} pairs | test {len(fixed['test']):,} pairs")

## 2. One model per fold

Each model trains with `train_ce` on the 4 other folds (no val curve, to save GPU time), then predicts its held-out fold, val and test with the class probabilities. Scores and probabilities are written as soon as a fold ends, so a stopped session keeps the folds already done.

In [ ]:
def subsample(df, frac, seed=SEED):
    keep = df['query_id'].drop_duplicates().sample(frac=frac, random_state=seed)
    return df[df['query_id'].isin(keep)]


runs = []
for k in (FOLDS if MODE != 'assemble' else []):
    tr, ho = train_all[train_all['fold'] != k], train_all[train_all['fold'] == k]
    if MODE == 'smoke':
        tr, ho = subsample(tr, 0.05), subsample(ho, 0.10)
    parts = {'oof': ho, 'val': fixed['val']} if MODE == 'smoke' else {'oof': ho, **fixed}
    parts = {s: p.reset_index(drop=True) for s, p in parts.items()}
    tr = tr.reset_index(drop=True)
    name = f"{'smoke_' if MODE == 'smoke' else ''}{NAME}_f{k}"
    print(f"\n==== {name}: train {tr['query_id'].nunique():,} queries / {len(tr):,} pairs | "
          + ' | '.join(f'{s} {len(p):,}' for s, p in parts.items()))

    log = RunLog(RES, {s: p[['example_id', 'query_id', 'esci_label']] for s, p in parts.items()})
    ref = REF if MODE == 'full' and log.load(REF, splits=['val']) else None
    model, tokenizer, info = train_ce(CFG, tr, SEED, print_every=100 if MODE == 'smoke' else 1000)

    t0 = time.time()
    torch.cuda.empty_cache()
    pred = {s: predict(model, tokenizer, p, CFG['max_len'], batch_size=CFG['eval_batch_size'], fp16=CFG['fp16'],
                       return_probs=True) for s, p in parts.items()}
    log.log(name, {s: v[0] for s, v in pred.items()},
            {'mode': MODE, **CFG, 'seed': SEED, 'fold': k, 'n_folds': N_FOLDS, 'train_folds': [f for f in range(N_FOLDS) if f != k],
             'train_queries': int(tr['query_id'].nunique()), 'train_pairs': len(tr),
             'gpu': torch.cuda.get_device_name(0), 'predict_minutes': round((time.time() - t0) / 60, 1), **info},
            compare_to=ref, blind=('test',), extra={s: v[1] for s, v in pred.items()})
    runs.append(name)
    del model, pred
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
if MODE == 'smoke':
    r = log.records[runs[0]]
    first, last = r['loss_log'][0]['loss'], r['loss_log'][-1]['loss']
    sc = pd.read_parquet(RES / 'scores' / f'{runs[0]}_oof.parquet')
    p_ok = list(sc.columns[-4:]) == PROBS and np.allclose(sc[PROBS].sum(axis=1), 1, atol=1e-3)
    full_h = r['steps'] / 0.05 / r['it_per_s'] / 3600
    checks = {'loss goes down': last < first,
              'val above random (0.742)': r['metrics']['val']['ndcg'] > 0.742,
              'held-out fold above random': r['metrics']['oof']['ndcg'] > 0.742,
              'p_E..p_I saved and sum to 1': p_ok}
    for c, ok in checks.items():
        print(f"{'OK   ' if ok else 'CHECK'} {c}")
    print(f"loss {first:.3f} -> {last:.3f} | val nDCG {r['metrics']['val']['ndcg']} | oof nDCG {r['metrics']['oof']['ndcg']} "
          f"| training per full fold ~{full_h:.2f} h")
    print('Smoke test passed: switch MODE to full.' if all(checks.values()) else 'Something failed: send me this output.')
elif MODE == 'full':
    display(log.table(('ndcg', 'ndcg_ci95', 'ndcg10', 'mrr_e', 'e_over_s')))
    log.print_paired()

## 3. Assemble the 5 folds (`MODE = 'assemble'`)

Attach the outputs of every version of this notebook (or a dataset with their `results/`). For each pair:
- **train**: score and probabilities of the one model that held its fold out;
- **val, test**: mean over the 5 fold models (probabilities averaged, score = expected gain of the averaged probabilities, which equals the mean score).

Checks: every train pair covered exactly once, by the model of its own fold; val/test covered by all 5 models. The table compares the train OOF nDCG with the val nDCG of the averaged models (similar values mean the OOF scores behave like the val scores the re-ranker will see) and with each single fold model on val.

In [ ]:
if MODE == 'assemble':
    def find_scores(k, split):
        hits = sorted(glob.glob(f'/kaggle/input/**/scores/{NAME}_f{k}_{split}.parquet', recursive=True)
                      + glob.glob(str(RES / 'scores' / f'{NAME}_f{k}_{split}.parquet')))
        assert hits, f'{NAME}_f{k}_{split}.parquet not found: attach the output of the version that ran fold {k}'
        return pd.read_parquet(hits[0])

    gain = {'E': 1.0, 'S': 0.1, 'C': 0.01, 'I': 0.0}
    oof = pd.concat([find_scores(k, 'oof').assign(fold=k) for k in range(N_FOLDS)], ignore_index=True)
    t = train_all[['example_id', 'fold']].merge(oof[['example_id', 'fold']], on='example_id', how='left', suffixes=('', '_model'))
    assert oof['example_id'].is_unique and len(oof) == len(train_all), 'train pairs missing or scored twice'
    assert (t['fold'] == t['fold_model']).all(), 'a train pair was scored by a model that saw its query'

    out, rows = [oof.drop(columns='fold').assign(split='train')], []
    for s in ['val', 'test']:
        per = [find_scores(k, s).set_index('example_id').loc[fixed[s]['example_id']] for k in range(N_FOLDS)]
        mean = sum(p[PROBS] for p in per) / N_FOLDS
        avg = mean.assign(query_id=per[0]['query_id'], score=mean @ np.array([gain[c[-1]] for c in PROBS]))
        out.append(avg.reset_index()[['example_id', 'query_id', 'score'] + PROBS].assign(split=s))
        if s == 'val':   # test stays blind
            for k, p in enumerate(per):
                rows.append({'scores': f'fold {k} model', 'split': 'val', **summarize(fixed['val'], p['score'].to_numpy())[0]})
            rows.append({'scores': 'mean of 5 models', 'split': 'val', **summarize(fixed['val'], avg['score'].to_numpy())[0]})
    tr_sorted = train_all[['example_id', 'query_id', 'esci_label']].merge(oof[['example_id', 'score']], on='example_id')
    rows.append({'scores': 'out-of-fold', 'split': 'train', **summarize(tr_sorted, tr_sorted['score'].to_numpy())[0]})
    display(pd.DataFrame(rows)[['scores', 'split', 'ndcg', 'ndcg_ci95', 'ndcg10', 'mrr_e', 'e_over_s']])

    table = pd.concat(out, ignore_index=True)
    table[['score'] + PROBS] = table[['score'] + PROBS].astype('float32')
    assert table['example_id'].is_unique and len(table) == len(pairs) and table[['score'] + PROBS].notna().all().all()
    (RES / 'oof').mkdir(parents=True, exist_ok=True)
    table.to_parquet(RES / 'oof' / 'ce_deberta_main_ce4.parquet', index=False)
    print('wrote', RES / 'oof' / 'ce_deberta_main_ce4.parquet', table.shape, table['split'].value_counts().to_dict())